# Raw data grab: every upstream source of every fire, one channel at a time

Pulls the original data behind the wildfire spread dataset
([Javyn123/wildfire-spread-2014-2025](https://huggingface.co/datasets/Javyn123/wildfire-spread-2014-2025)) into a
Hugging Face dataset repo you choose. Each fire gets the records, values and grid of every source, cut to its box
and dates. The code is `pipeline/raw_grab.py` on the `data-grab` branch of
[WildfireRepo1](https://github.com/GT-Fire-Research-and-Innovation/WildfireRepo1).

**Before you run**
- Settings → Internet: **on**.
- Add-ons → Secrets:
  - `HF_TOKEN`: a Hugging Face token with write access
  - `FIRMS_MAP_KEY`: only for `firms`; free at https://firms.modaps.eosdis.nasa.gov/api/map_key/
  - `LANDFIRE_EMAIL`: only for `landfire`; the LANDFIRE service needs a contact email

**How it runs**
- Run one channel per session. Each batch (about 2 GB or 100 fires) is uploaded as one commit, then deleted from
  disk.
- Before Kaggle's 12-hour limit the run stops cleanly. Run the notebook again and it continues where it stopped:
  fires already in the repo's manifest are skipped.
- Suggested order, from cheap to heavy: `history`, `perimeters`, `firms`, `gridmet`, `ics209`, `suppression`,
  `tessera`, `terrain`, `landfire`, `goes`, `hrrr`, `osm`. `goes`, `hrrr` and `osm` take several sessions each.

In [ ]:
# ---- settings
HF_REPO = "your-name/wildfire-raw-2014-2025"   # target dataset repo (created if missing)
PRIVATE = True                                 # only used when the repo is created
CHANNEL = "history"                            # one of CHANNELS below
FIRES = "built"                                # built (1,841 fires with a built folder) | keep | all
WORK = "/tmp/raw"                              # scratch: staging, batches, one-time downloads (not /kaggle/working)
MAX_HOURS = 11.5                               # stop cleanly before Kaggle's 12 h session limit
BATCH_GB = 2.0

CHANNELS = ["history", "perimeters", "firms", "gridmet", "ics209", "suppression", "tessera", "terrain",
            "landfire", "goes", "hrrr", "osm"]

In [ ]:
# ---- the project code, data-grab branch
!git clone -q --depth 1 -b data-grab https://github.com/GT-Fire-Research-and-Innovation/WildfireRepo1.git 2>/dev/null || (cd WildfireRepo1 && git pull -q)
!cd WildfireRepo1 && git log -1 --format='%h %s'

In [ ]:
# ---- packages (Kaggle has numpy, pandas, geopandas, rasterio, xarray, netCDF4)
!pip install -q pyogrio pygrib "huggingface_hub>=0.24"
import shutil
if CHANNEL == "osm" and not shutil.which("osmium"):
    !apt-get -qq update && apt-get -qq install -y osmium-tool aria2 > /dev/null

In [ ]:
# ---- secrets -> environment (never printed)
import os
try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    for _k in ("HF_TOKEN", "FIRMS_MAP_KEY", "LANDFIRE_EMAIL"):
        try:
            os.environ[_k] = _s.get_secret(_k)
        except Exception:
            pass
except ImportError:
    pass                                       # not on Kaggle: use the existing environment
print({k: ("set" if os.environ.get(k) else "missing") for k in ("HF_TOKEN", "FIRMS_MAP_KEY", "LANDFIRE_EMAIL")})

In [ ]:
import subprocess, sys

def grab(channel, **opts):
    """Run one channel (resumes from the repo's manifest). opts: ids=[...], limit=N, dry_run=True, fires='keep', ..."""
    cmd = [sys.executable, "-u", "WildfireRepo1/pipeline/raw_grab.py", "--channel", channel, "--hf-repo", HF_REPO,
           "--work", WORK, "--fires", opts.pop("fires", FIRES), "--max-hours", str(opts.pop("max_hours", MAX_HOURS)),
           "--batch-gb", str(opts.pop("batch_gb", BATCH_GB))]
    if PRIVATE:
        cmd.append("--private")
    for k, v in opts.items():
        flag = "--" + k.replace("_", "-")
        if v is True:
            cmd.append(flag)
        elif isinstance(v, (list, tuple)):
            cmd += [flag, *map(str, v)]
        elif v not in (None, False):
            cmd += [flag, str(v)]
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    return p.wait()

def status():
    subprocess.run([sys.executable, "WildfireRepo1/pipeline/raw_grab.py", "--status", "--hf-repo", HF_REPO])

Optional check first: one fire, nothing uploaded. Camp Fire 2018 is `F2003403265`.

In [ ]:
# grab(CHANNEL, ids=["F2003403265"], dry_run=True)
# !find /tmp/raw/batch -type f | head

## Run the channel

In [ ]:
grab(CHANNEL)

Or run every channel in order in one session (each one resumes on the next run):

In [ ]:
# import time
# t0 = time.time()
# for ch in CHANNELS:
#     left = MAX_HOURS - (time.time() - t0) / 3600
#     if left < 0.5:
#         break
#     grab(ch, max_hours=left)

## Progress

In [ ]:
status()